# dataset 入门

文本需要先由分词工具被分解成一个个独立的词元。作为快速入门，你可以加载 Microsoft Research Paraphrase Corpus (MRPC)训练数据集，以此来训练模型，从而判断两句话是否表达的是相同的意思。

1. 要加载 MRPC 数据集，需使用 load_dataset()函数，并指定数据集的名称、配置信息（并非所有数据集都需要配置信息），以及数据集的划分方式：

In [ ]:
from datasets import load_dataset

dataset = load_dataset("nyu-mll/glue", "mrpc", split="train")

2. 接下来，从🤗 Transformers 库中加载预训练好的 BERT 模型及其对应的分词器。在加载模型后，出现关于某些权重未被初始化的警告是很正常的。这是预料之中的情况，因为你正在将这个模型用于与原本任务不同的训练场景中。

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

3. 编写一个函数来对数据集进行分词处理。同时，还需要将文本截断或补齐，使其变成结构整齐的矩形张量形式。该分词函数会在数据集中生成三列新的数据： input_ids 、 token_type_ids 和 attention_mask 。这三列数据将作为模型的输入。

使用 map()函数，可以通过将分词函数应用于数据集中的多个样本，从而加快处理速度。

In [ ]:
def encode(examples):
    return tokenizer(examples["sentence1"], examples["sentence2"], truncation=True, padding="max_length")

dataset = dataset.map(encode, batched=True)
dataset[0]

4. 将 label 列重命名为 labels ，这正是 BertForSequenceClassification 模型所期望的输入列名。

In [ ]:
dataset = dataset.map(lambda examples: {"labels": examples["label"]}, batched=True)

5. 请根据您所使用的机器学习框架来设置数据集的格式。

In [ ]:
import torch

dataset = dataset.select_columns(["input_ids", "token_type_ids", "attention_mask", "labels"])
dataset = dataset.with_format(type="torch")
dataloader = torch.utils.data.DataLoader(dataset, batch_size=32)